*EXECUTION TIME*

In [41]:
import time

start_time = time.time()

**IMPORT LIBRARIES**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

**CREATE SPARK SESSION**

In [2]:
spark = (
    SparkSession.builder
    .appName("Retail Sales ETL Pipeline")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark Version :", spark.version)

Spark Version : 4.2.0


**READ DATASET**

In [3]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")   # IMPORTANT
    .option("multiLine", "true")
    .option("escape", '"')
    .option("quote", '"')
    .load(r"D:\Analytics\retail_sales_ETL_pipeline\data\raw\superstore.csv")
)

**BASIC DATA EXPLORATION**

In [4]:
print("="*60)
print("DATASET OVERVIEW")
print("="*60)

print(f"Total Rows    : {df.count()}")
print(f"Total Columns : {len(df.columns)}")

print("\nColumn Names:")
print(df.columns)

print("\nSchema:")
df.printSchema()

DATASET OVERVIEW
Total Rows    : 9994
Total Columns : 24

Column Names:
['ID', 'Order_id', 'Order_Date', 'Ship _Date', 'Ship_Mode', 'Customer_id', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'user_id', 'state_id', 'order_s']

Schema:
root
 |-- ID: string (nullable = true)
 |-- Order_id: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship _Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_id: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: s

**Standardize Column Name**

In [5]:
# Standardize column names

new_columns = []

for column in df.columns:
    column = column.strip()                  # Remove leading/trailing spaces
    column = column.replace(" ", "_")        # Replace spaces with _
    column = column.replace("__", "_")       # Remove double underscores
    column = column.lower()                  # Convert to lowercase
    new_columns.append(column)

df = df.toDF(*new_columns)

print(df.columns)

['id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'user_id', 'state_id', 'order_s']


*Convert Datatypes*

In [6]:
from pyspark.sql.functions import col, to_date

df = (
    df
    .withColumn("sales", col("sales").cast("double"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("discount", col("discount").cast("double"))
    .withColumn("profit", col("profit").cast("double"))
    .withColumn("order_date", to_date(try_to_timestamp(col("order_date"))))
)

In [7]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- order_s: string (nullable = true)



**Preview Dataset**

In [8]:
df.select("order_date").show(10, truncate=False)

+----------+
|order_date|
+----------+
|2023-11-08|
|2023-11-08|
|2023-06-12|
|2022-10-11|
|2022-10-11|
|2021-06-09|
|2021-06-09|
|2021-06-09|
|2021-06-09|
|2021-06-09|
+----------+
only showing top 10 rows


In [9]:
df.show(10, truncate=False)

+---+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+-------+--------+-------+
|id |order_id      |order_date|ship_date |ship_mode     |customer_id|customer_name  |segment  |country      |city           |state     |postal_code|region|product_id     |category       |sub_category|product_name                                                    |sales   |quantity|discount|profit  |user_id|state_id|order_s|
+---+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+-------+--------+-------+
|1  |CA-2023-152156

**Data Quality Checks**

In [10]:
from pyspark.sql.functions import col, when, count

print("Missing Values in Each Column")

missing_df = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_df.show(vertical=True, truncate=False)

Missing Values in Each Column
-RECORD 0-------------
 id            | 0    
 order_id      | 0    
 order_date    | 1    
 ship_date     | 0    
 ship_mode     | 0    
 customer_id   | 0    
 customer_name | 0    
 segment       | 0    
 country       | 0    
 city          | 0    
 state         | 0    
 postal_code   | 0    
 region        | 0    
 product_id    | 0    
 category      | 0    
 sub_category  | 0    
 product_name  | 0    
 sales         | 0    
 quantity      | 0    
 discount      | 0    
 profit        | 0    
 user_id       | 9993 
 state_id      | 9992 
 order_s       | 9992 



**Duplicate Records**

In [11]:
total_rows = df.count()

unique_rows = df.dropDuplicates().count()

duplicates = total_rows - unique_rows

print(f"Total Rows      : {total_rows}")
print(f"Unique Rows     : {unique_rows}")
print(f"Duplicate Rows  : {duplicates}")

Total Rows      : 9994
Unique Rows     : 9994
Duplicate Rows  : 0


**Unique Values**

In [12]:
print("Categories")
df.select("Category").distinct().show()

print("Sub Categories")
df.select("Sub_Category").distinct().show()

print("Segments")
df.select("Segment").distinct().show()

print("Ship Modes")
df.select("Ship_Mode").distinct().show()

Categories
+---------------+
|       Category|
+---------------+
|Office Supplies|
|      Furniture|
|     Technology|
+---------------+

Sub Categories
+------------+
|Sub_Category|
+------------+
|   Envelopes|
|         Art|
|      Chairs|
| Furnishings|
|    Supplies|
|   Fasteners|
|     Binders|
|   Bookcases|
|      Labels|
|       Paper|
| Accessories|
|     Copiers|
|      Phones|
|    Machines|
|     Storage|
|  Appliances|
|      Tables|
+------------+

Segments
+-----------+
|    Segment|
+-----------+
|   Consumer|
|Home Office|
|  Corporate|
+-----------+

Ship Modes
+--------------+
|     Ship_Mode|
+--------------+
|   First Class|
|      Same Day|
|  Second Class|
|Standard Class|
+--------------+



**Business Validation**

In [13]:
print("Negative Sales")
df.filter(col("Sales") < 0).show()

print("Negative Quantity")
df.filter(col("Quantity") < 0).show()

print("Discount > 1")
df.filter(col("Discount") > 1).show()

print("Profit < 0")
df.filter(col("Profit") < 0).show()

Negative Sales
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+
| id|order_id|order_date|ship_date|ship_mode|customer_id|customer_name|segment|country|city|state|postal_code|region|product_id|category|sub_category|product_name|sales|quantity|discount|profit|user_id|state_id|order_s|
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+

Negative Quantity
+---+--------+----------+---------+---------+-----------+-------------+-------+-------

**Count Distinct Values**

In [14]:
print("Customers :", df.select("Customer_ID").distinct().count())

print("Products :", df.select("Product_ID").distinct().count())

print("Orders :", df.select("Order_ID").distinct().count())

print("Cities :", df.select("City").distinct().count())

print("States :", df.select("State").distinct().count())

Customers : 793
Products : 1861
Orders : 5009
Cities : 531
States : 49


**DATA CLEANING AND TRANSFORMATION**

*1. REMOVE DUPLICATES*

In [15]:
print(f"Rows Before Removing Duplicates : {df.count()}")

df = df.dropDuplicates()

print(f"Rows After Removing Duplicates  : {df.count()}")

Rows Before Removing Duplicates : 9994
Rows After Removing Duplicates  : 9994


*2. MISSING VALUE ANALYSIS*

In [16]:
from pyspark.sql.functions import col, when, count

missing_df = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_df.show(vertical=True, truncate=False)

-RECORD 0-------------
 id            | 0    
 order_id      | 0    
 order_date    | 1    
 ship_date     | 0    
 ship_mode     | 0    
 customer_id   | 0    
 customer_name | 0    
 segment       | 0    
 country       | 0    
 city          | 0    
 state         | 0    
 postal_code   | 0    
 region        | 0    
 product_id    | 0    
 category      | 0    
 sub_category  | 0    
 product_name  | 0    
 sales         | 0    
 quantity      | 0    
 discount      | 0    
 profit        | 0    
 user_id       | 9993 
 state_id      | 9992 
 order_s       | 9992 



*3. Standardize Text Columns*

In [17]:
from pyspark.sql.functions import trim, initcap

text_columns = [
    "customer_name",
    "city",
    "state",
    "region",
    "category",
    "sub_category",
    "product_name",
    "segment",
    "ship_mode"
]

for column in text_columns:
    df = df.withColumn(column, initcap(trim(col(column))))

*Business Rule Validation*

*1. Sales should never be negative*

In [18]:
df.filter(col("sales") < 0).show()

+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+
| id|order_id|order_date|ship_date|ship_mode|customer_id|customer_name|segment|country|city|state|postal_code|region|product_id|category|sub_category|product_name|sales|quantity|discount|profit|user_id|state_id|order_s|
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+



*Quantity should never be negative*

In [19]:
df.filter(col("quantity") < 0).show()

+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+
| id|order_id|order_date|ship_date|ship_mode|customer_id|customer_name|segment|country|city|state|postal_code|region|product_id|category|sub_category|product_name|sales|quantity|discount|profit|user_id|state_id|order_s|
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+



*Discount should be between 0 and 1*

In [20]:
df.filter(
    (col("discount") < 0) |
    (col("discount") > 1)
).show()

+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+
| id|order_id|order_date|ship_date|ship_mode|customer_id|customer_name|segment|country|city|state|postal_code|region|product_id|category|sub_category|product_name|sales|quantity|discount|profit|user_id|state_id|order_s|
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+
+---+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------+--------+-------+



**CREATE BUSINESS FEATURES**

*SHIPPING DAYS*

In [21]:
from pyspark.sql.functions import datediff

df = df.withColumn(
    "shipping_days",
    datediff(col("ship_date"), col("order_date"))
)

*ORDER YEAR*

In [22]:
from pyspark.sql.functions import year

df = df.withColumn(
    "order_year",
    year(col("order_date"))
)

*ORDER MONTH*

In [23]:
from pyspark.sql.functions import month

df = df.withColumn(
    "order_month",
    month(col("order_date"))
)

*ORDER QUARTER*

In [24]:
from pyspark.sql.functions import quarter

df = df.withColumn(
    "order_quarter",
    quarter(col("order_date"))
)

*PROFIT MARGIN*

In [25]:
from pyspark.sql.functions import round

df = df.withColumn(
    "profit_margin",
    round((col("profit") / col("sales")) * 100, 2)
)

*Validate New Columns*

In [26]:
df.select(
    "sales",
    "profit",
    "profit_margin",
    "shipping_days",
    "order_year",
    "order_month",
    "order_quarter"
).show(10, truncate=False)

+-------+-------+-------------+-------------+----------+-----------+-------------+
|sales  |profit |profit_margin|shipping_days|order_year|order_month|order_quarter|
+-------+-------+-------------+-------------+----------+-----------+-------------+
|89.584 |4.4792 |5.0          |7            |2022      |1          |1            |
|8.26   |3.7996 |46.0         |5            |2023      |4          |2            |
|12.22  |3.666  |30.0         |2            |2023      |9          |3            |
|122.48 |0.0    |0.0          |1            |2021      |12         |4            |
|21.8   |6.104  |28.0         |4            |2023      |11         |4            |
|18.16  |6.583  |36.25        |3            |2023      |10         |4            |
|18.97  |9.1056 |48.0         |2            |2023      |9          |3            |
|7.42   |3.71   |50.0         |6            |2024      |10         |4            |
|159.984|11.9988|7.5          |5            |2022      |12         |4            |
|64.

In [27]:
#CHECH FINAL SCHEMA
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- order_s: string (nullable = true)
 |-- shipping_days: integer (nullab

In [28]:
df.cache()

print("DataFrame Cached Successfully")

DataFrame Cached Successfully


**SECTION 1: SPARK SQL ANALYSIS**

In [29]:
df.createOrReplaceTempView("retail_sales")

**1. TOTAL SALES BY CATEGORY**

In [30]:
spark.sql("""
SELECT
    category,
    ROUND(SUM(sales),2) AS total_sales
FROM retail_sales
GROUP BY category
ORDER BY total_sales DESC
""").show()

+---------------+-----------+
|       category|total_sales|
+---------------+-----------+
|     Technology|  836154.03|
|      Furniture|   741999.8|
|Office Supplies|  719047.03|
+---------------+-----------+



**2. Total Profit by Category**

In [31]:
spark.sql("""
SELECT
    category,
    ROUND(SUM(profit),2) AS total_profit
FROM retail_sales
GROUP BY category
ORDER BY total_profit DESC
""").show()

+---------------+------------+
|       category|total_profit|
+---------------+------------+
|     Technology|   145454.95|
|Office Supplies|    122490.8|
|      Furniture|    18451.27|
+---------------+------------+



**3. Top 10 Product by Sales**

In [32]:
spark.sql("""
SELECT
    product_name,
    ROUND(SUM(sales),2) AS total_sales
FROM retail_sales
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 10
""").show()

+--------------------+-----------+
|        product_name|total_sales|
+--------------------+-----------+
|Canon Imageclass ...|   61599.82|
|Fellowes Pb500 El...|   27453.38|
|Cisco Telepresenc...|   22638.48|
|Hon 5400 Series T...|   21870.58|
|Gbc Docubind Tl30...|   19823.48|
|Gbc Ibimaster 500...|    19024.5|
|Hewlett Packard L...|   18839.69|
|Hp Designjet T520...|    18374.9|
|Gbc Docubind P400...|   17965.07|
|High Speed Automa...|   17030.31|
+--------------------+-----------+



**4. Top 10 State by Sales**

In [33]:
spark.sql("""
SELECT
    state,
    ROUND(SUM(sales),2) AS total_sales
FROM retail_sales
GROUP BY state
ORDER BY total_sales DESC
LIMIT 10
""").show()

+------------+-----------+
|       state|total_sales|
+------------+-----------+
|  California|  457687.63|
|    New York|  310876.27|
|       Texas|  170188.05|
|  Washington|  138641.27|
|Pennsylvania|  116511.91|
|     Florida|   89473.71|
|    Illinois|    80166.1|
|        Ohio|   78258.14|
|    Michigan|   76269.61|
|    Virginia|   70636.72|
+------------+-----------+



**5. Sales by Region**

In [34]:
spark.sql("""
SELECT
    region,
    ROUND(SUM(sales),2) AS total_sales
FROM retail_sales
GROUP BY region
ORDER BY total_sales DESC
""").show()

+-------+-----------+
| region|total_sales|
+-------+-----------+
|   West|  725457.82|
|   East|  678781.24|
|Central|  501239.89|
|  South|  391721.91|
+-------+-----------+



**6. Customer Segment Analysis**

In [35]:
spark.sql("""
SELECT
    segment,
    COUNT(DISTINCT customer_id) AS customers,
    ROUND(SUM(sales),2) AS revenue
FROM retail_sales
GROUP BY segment
ORDER BY revenue DESC
""").show()

+-----------+---------+----------+
|    segment|customers|   revenue|
+-----------+---------+----------+
|   Consumer|      409|1161401.35|
|  Corporate|      236| 706146.37|
|Home Office|      148| 429653.15|
+-----------+---------+----------+



**7. Monthly Revenue**

In [36]:
spark.sql("""
SELECT
    order_year,
    order_month,
    ROUND(SUM(sales),2) AS revenue
FROM retail_sales
GROUP BY order_year, order_month
ORDER BY order_year, order_month
""").show()

+----------+-----------+--------+
|order_year|order_month| revenue|
+----------+-----------+--------+
|      NULL|       NULL|   111.1|
|      2021|          1| 14236.9|
|      2021|          2| 4519.89|
|      2021|          3|55691.01|
|      2021|          4|28295.35|
|      2021|          5|23648.29|
|      2021|          6|34595.13|
|      2021|          7|33946.39|
|      2021|          8|27909.47|
|      2021|          9|81777.35|
|      2021|         10|31453.39|
|      2021|         11|78628.72|
|      2021|         12|69545.62|
|      2022|          1|18174.08|
|      2022|          2|11951.41|
|      2022|          3|38726.25|
|      2022|          4|34195.21|
|      2022|          5|30131.69|
|      2022|          6|24797.29|
|      2022|          7|28765.33|
+----------+-----------+--------+
only showing top 20 rows


**8. Average Profit Margin by Category**

In [37]:
spark.sql("""
SELECT
    category,
    ROUND(AVG(profit_margin),2) AS avg_profit_margin
FROM retail_sales
GROUP BY category
ORDER BY avg_profit_margin DESC
""").show()

+---------------+-----------------+
|       category|avg_profit_margin|
+---------------+-----------------+
|     Technology|            15.61|
|Office Supplies|             13.8|
|      Furniture|             3.88|
+---------------+-----------------+



**9. Shipping Performance**

In [38]:
spark.sql("""
SELECT
    ship_mode,
    ROUND(AVG(shipping_days),2) AS avg_shipping_days
FROM retail_sales
GROUP BY ship_mode
ORDER BY avg_shipping_days
""").show()

+--------------+-----------------+
|     ship_mode|avg_shipping_days|
+--------------+-----------------+
|Standard Class|           -10.42|
|  Second Class|            -4.65|
|      Same Day|             0.04|
|   First Class|             2.18|
+--------------+-----------------+



**Average Discount by Category**

In [39]:
spark.sql("""
SELECT
    category,
    ROUND(AVG(discount),2) AS avg_discount
FROM retail_sales
GROUP BY category
ORDER BY avg_discount DESC
""").show()

+---------------+------------+
|       category|avg_discount|
+---------------+------------+
|      Furniture|        0.17|
|Office Supplies|        0.16|
|     Technology|        0.13|
+---------------+------------+



*End Time*

In [42]:
end_time = time.time()

print("="*70)
print("RETAIL SALES ETL PIPELINE EXECUTED SUCCESSFULLY")
print("="*70)
print(f"Execution Time : {end_time - start_time:.2f} seconds")

RETAIL SALES ETL PIPELINE EXECUTED SUCCESSFULLY
Execution Time : 28.74 seconds


**PIPELINE SUMMARY**

In [43]:
print("="*70)
print("             RETAIL SALES ETL PIPELINE SUMMARY")
print("="*70)

print(f"Total Records Processed : {df.count()}")
print(f"Total Columns           : {len(df.columns)}")
print("Data Cleaning           : Completed")
print("Feature Engineering     : Completed")
print("Spark SQL Queries       : 10 Executed")
print("Export Layer            : Pending (Windows Hadoop Configuration)")
print("Pipeline Status         : SUCCESS")

             RETAIL SALES ETL PIPELINE SUMMARY
Total Records Processed : 9994
Total Columns           : 29
Data Cleaning           : Completed
Feature Engineering     : Completed
Spark SQL Queries       : 10 Executed
Export Layer            : Pending (Windows Hadoop Configuration)
Pipeline Status         : SUCCESS
